# EL-GHALI MOHAMED


### Analyse en Composantes Principales (PCA)

L'Analyse en Composantes Principales est une transformation mathématique. L'idée est de créer de nouvelles caractéristiques (les Composantes Principales) qui sont des combinaisons linéaires des anciennes, mais tournées dans la direction où les données varient le plus.

 Les 4 étapes mathématiques

1. **Standardisation** : Centrer et réduire les données.
$$Z = \frac{X - \mu}{\sigma}$$


2. **Matrice de Covariance** : Calculer comment les variables varient ensemble.
$$\Sigma = \frac{1}{n-1} Z^T Z$$


3. **Décomposition propre** : Trouver les vecteurs propres ($v$) et les valeurs propres ($\lambda$) de la matrice de covariance.
$$\Sigma v = \lambda v$$


4. **Projection** : Créer la nouvelle matrice réduite.




###  Jeu de données

In [1]:
import numpy as np
import pandas as pd

# Création d'un dataset fictif (Taille en cm, Poids en kg, Pointure)
donnees = {
    'Taille': [160, 165, 170, 175, 180, 185, 190],
    'Poids': [60, 65, 70, 74, 80, 85, 92],
    'Pointure': [38, 39, 41, 42, 43, 44, 46]
}
df = pd.DataFrame(donnees)

# On convertit le DataFrame Pandas en matrice Numpy pour les calculs mathématiques
X = df.values

print("=== Données d'origine (3 Dimensions) ===")
print(df)
print(f"Forme de la matrice : {X.shape} (7 lignes, 3 colonnes)")

=== Données d'origine (3 Dimensions) ===
   Taille  Poids  Pointure
0     160     60        38
1     165     65        39
2     170     70        41
3     175     74        42
4     180     80        43
5     185     85        44
6     190     92        46
Forme de la matrice : (7, 3) (7 lignes, 3 colonnes)


### Standardisation (Centrer et Réduire)

In [2]:
def standardiser_donnees(X):
    """Centre (moyenne = 0) et réduit (écart-type = 1) les données."""
    # Calcul de la moyenne pour chaque colonne (axis=0)
    moyennes = np.mean(X, axis=0)

    # Calcul de l'écart-type pour chaque colonne
    ecarts_types = np.std(X, axis=0)

    # Application de la formule : (Valeur - Moyenne) / Ecart-type
    X_standardise = (X - moyennes) / ecarts_types

    return X_standardise

X_std = standardiser_donnees(X)

print("Moyennes après standardisation (doivent être proches de 0) :", np.mean(X_std, axis=0))
print("Écarts-types après standardisation (doivent être de 1) :", np.std(X_std, axis=0))

Moyennes après standardisation (doivent être proches de 0) : [0.00000000e+00 3.17206578e-16 1.17366434e-15]
Écarts-types après standardisation (doivent être de 1) : [1. 1. 1.]


### Matrice de Covariance et Valeurs/Vecteurs Propres
- **La Matrice de Covariance** nous indique quelles caractéristiques évoluent dans le même sens.

- **Les Vecteurs Propres** représentent les nouvelles "directions" (axes) de nos données.

- **Les Valeurs Propres** représentent la quantité d'information (variance) contenue dans chacune de ces directions.

In [3]:
def calculer_composantes_principales(X_std):
    """Calcule la matrice de covariance et en extrait les composantes principales."""
    n = X_std.shape[0] # Nombre de lignes

    #  Calcul de la Matrice de Covariance
    # Formule : (X transposé multiplié par X) / (n - 1)
    matrice_covariance = (X_std.T @ X_std) / (n - 1)

    #  Calcul des Valeurs propres et Vecteurs propres
    # on utilise la fonction d'algèbre linéaire de Numpy.
    valeurs_propres, vecteurs_propres = np.linalg.eig(matrice_covariance)

    #  Trier les composantes de la plus importante à la moins importante
    # On obtient les indices du tri décroissant des valeurs propres
    indices_tries = np.argsort(valeurs_propres)[::-1]

    valeurs_propres_triees = valeurs_propres[indices_tries]
    vecteurs_propres_tries = vecteurs_propres[:, indices_tries]

    return valeurs_propres_triees, vecteurs_propres_tries

valeurs_propres, vecteurs_propres = calculer_composantes_principales(X_std)

print("=== Valeurs Propres (Variance expliquée par chaque axe) ===")
print(valeurs_propres)

# Calcul du pourcentage d'information (variance) conservé par chaque axe
variance_totale = sum(valeurs_propres)
pourcentage_variance = [(i / variance_totale) * 100 for i in valeurs_propres]

print("\nPourcentage d'information par composante :")
for i, pourcentage in enumerate(pourcentage_variance):
    print(f"Composante {i+1} : {pourcentage:.2f}% de l'information")

=== Valeurs Propres (Variance expliquée par chaque axe) ===
[3.48862294e+00 8.65032035e-03 2.72673748e-03]

Pourcentage d'information par composante :
Composante 1 : 99.67% de l'information
Composante 2 : 0.25% de l'information
Composante 3 : 0.08% de l'information


### Projection : Réduction de la dimensionnalité

In [4]:
def projeter_donnees(X_std, vecteurs_propres, nb_dimensions):
    """Projette les données sur les k premières composantes principales."""
    # On sélectionne seulement les 'k' premiers vecteurs propres
    # Cela devient notre Matrice de Projection (W)
    matrice_projection = vecteurs_propres[:, 0:nb_dimensions]

    # On multiplie nos données standardisées par cette matrice de projection
    X_reduit = X_std @ matrice_projection

    return X_reduit

# Nous voulons passer de 3 dimensions à 2 dimensions
K = 2

X_pca = projeter_donnees(X_std, vecteurs_propres, nb_dimensions=K)

# Création d'un  DataFrame pour le résultat final
df_pca = pd.DataFrame(X_pca, columns=[f'Composante Principale {i+1}' for i in range(K)])

print(f"=== Nouvelles Données Réduites ({K} Dimensions) ===")
print(df_pca)
print(f"\nNouvelle forme de la matrice : {X_pca.shape} (7 lignes, {K} colonnes)")

=== Nouvelles Données Réduites (2 Dimensions) ===
   Composante Principale 1  Composante Principale 2
0                 2.563130                -0.015255
1                 1.775088                -0.098494
2                 0.764084                 0.133980
3                 0.031298                 0.090219
4                -0.812000                -0.032497
5                -1.600042                -0.115735
6                -2.721558                 0.037782

Nouvelle forme de la matrice : (7, 2) (7 lignes, 2 colonnes)
